# Session 3: LangGraph Orchestration & Workflow Design (Student Notebook)

Welcome to Session 3! In this notebook, you will learn LangGraph — a framework for building stateful, multi-step agentic workflows as directed graphs. You will model complex agent behaviors using nodes, edges, and shared state, enabling conditional routing, iterative refinement, and human-in-the-loop patterns.

## Learning Objectives

By the end of this session, you will be able to:

1. **Create StateGraphs** with typed state schemas
2. **Define nodes and edges** to model workflow steps
3. **Add conditional edges** for dynamic routing based on LLM decisions
4. **Build cyclic workflows** for iterative refinement
5. **Implement a ReAct agent** using the Reason-Act-Observe loop
6. **Add human-in-the-loop** checkpoints for oversight

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: LangGraph Basics — Creating a Simple StateGraph

A StateGraph has three core components:
- **State**: A TypedDict that holds all data flowing through the graph
- **Nodes**: Python functions that read and update state
- **Edges**: Connections that define the execution order

In [ ]:
# Demo 1 - LangGraph Basics: Creating a Simple StateGraph

# Step 1: Define the state schema
class SimpleState(TypedDict):
    input_text: str
    uppercase_text: str
    word_count: int
    result: str

# Step 2: Define node functions (each takes state, returns partial state update)
def uppercase_node(state: SimpleState) -> dict:
    """Convert input text to uppercase."""
    return {"uppercase_text": state["input_text"].upper()}

def count_words_node(state: SimpleState) -> dict:
    """Count words in the input text."""
    return {"word_count": len(state["input_text"].split())}

def format_result_node(state: SimpleState) -> dict:
    """Format the final result."""
    return {"result": f"Text: {state['uppercase_text']} | Words: {state['word_count']}"}

# Step 3: Build the graph
graph = StateGraph(SimpleState)

# Add nodes
graph.add_node("uppercase", uppercase_node)
graph.add_node("count_words", count_words_node)
graph.add_node("format_result", format_result_node)

# Add edges (linear flow)
graph.set_entry_point("uppercase")
graph.add_edge("uppercase", "count_words")
graph.add_edge("count_words", "format_result")
graph.add_edge("format_result", END)

# Step 4: Compile and run
app = graph.compile()

result = app.invoke({"input_text": "Hello world from LangGraph", "uppercase_text": "", "word_count": 0, "result": ""})
print("Graph result:")
print(f"  Input    : {result['input_text']}")
print(f"  Uppercase: {result['uppercase_text']}")
print(f"  Words    : {result['word_count']}")
print(f"  Result   : {result['result']}")

### Demo 2: Adding Conditional Edges for Routing

Conditional edges let the graph take different paths based on the current state. This is how agents make decisions — an LLM evaluates the situation and routes to the appropriate next step.

In [ ]:
# Demo 2 - Adding Conditional Edges for Routing

class RouterState(TypedDict):
    query: str
    category: str
    response: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def classify_node(state: RouterState) -> dict:
    """Classify the query into a category."""
    response = llm.invoke([
        SystemMessage(content="Classify the query into exactly one category: 'technical', 'creative', or 'factual'. Respond with just the category word."),
        HumanMessage(content=state["query"])
    ])
    category = response.content.strip().lower()
    print(f"  Classified as: {category}")
    return {"category": category}

def technical_node(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a technical expert. Give a precise, detailed answer."),
        HumanMessage(content=state["query"])
    ])
    return {"response": f"[TECHNICAL] {response.content}"}

def creative_node(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a creative writer. Give an imaginative, engaging answer."),
        HumanMessage(content=state["query"])
    ])
    return {"response": f"[CREATIVE] {response.content}"}

def factual_node(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a factual encyclopedia. Give a brief, accurate answer."),
        HumanMessage(content=state["query"])
    ])
    return {"response": f"[FACTUAL] {response.content}"}

# Routing function
def route_by_category(state: RouterState) -> str:
    category = state["category"]
    if "technical" in category:
        return "technical"
    elif "creative" in category:
        return "creative"
    else:
        return "factual"

# Build graph
graph = StateGraph(RouterState)
graph.add_node("classify", classify_node)
graph.add_node("technical", technical_node)
graph.add_node("creative", creative_node)
graph.add_node("factual", factual_node)

graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_by_category, {
    "technical": "technical",
    "creative": "creative",
    "factual": "factual"
})
graph.add_edge("technical", END)
graph.add_edge("creative", END)
graph.add_edge("factual", END)

app = graph.compile()

# Test with different queries
queries = [
    "How does a hash table handle collisions?",
    "Write a poem about the ocean",
    "What is the capital of France?"
]

for query in queries:
    print(f"\nQuery: {query}")
    result = app.invoke({"query": query, "category": "", "response": ""})
    print(f"Response: {result['response'][:200]}...")

### Demo 3: Building a ReAct Agent with LangGraph

The ReAct (Reason + Act) pattern is the most common agentic loop:
1. **Think** — The LLM reasons about what to do
2. **Act** — Execute a tool/action
3. **Observe** — Process the result
4. **Repeat** until the task is complete

In [ ]:
# Demo 3 - Building a ReAct Agent with LangGraph

class ReActState(TypedDict):
    question: str
    thoughts: list[str]
    actions: list[str]
    observations: list[str]
    final_answer: str
    step_count: int

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Simulated tools
def search(query):
    data = {
        "python creator": "Guido van Rossum created Python in 1991.",
        "langchain": "LangChain is a framework for building LLM applications.",
        "langgraph": "LangGraph builds on LangChain for stateful multi-step workflows."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No results found for: {query}"

def think_node(state: ReActState) -> dict:
    """LLM thinks about what to do next."""
    context = ""
    for i in range(len(state["actions"])):
        context += f"Action: {state['actions'][i]}\nObservation: {state['observations'][i]}\n"
    
    response = llm.invoke([
        SystemMessage(content="You are a reasoning agent. Given the question and any previous observations, decide: either search for more info (respond: ACTION: search '<query>') or give a final answer (respond: FINAL ANSWER: <answer>)."),
        HumanMessage(content=f"Question: {state['question']}\n\n{context}\nWhat should I do next?")
    ])
    thought = response.content
    print(f"  Think: {thought[:100]}...")
    return {"thoughts": state["thoughts"] + [thought]}

def act_node(state: ReActState) -> dict:
    """Execute the action from the latest thought."""
    latest_thought = state["thoughts"][-1]
    
    if "FINAL ANSWER:" in latest_thought:
        answer = latest_thought.split("FINAL ANSWER:")[-1].strip()
        return {"final_answer": answer, "step_count": state["step_count"] + 1}
    
    if "ACTION: search" in latest_thought:
        query = latest_thought.split("search")[-1].strip().strip("'\"")
        observation = search(query)
        print(f"  Act: search('{query}') -> {observation[:60]}...")
        return {
            "actions": state["actions"] + [f"search('{query}')"],
            "observations": state["observations"] + [observation],
            "step_count": state["step_count"] + 1
        }
    
    return {"final_answer": latest_thought, "step_count": state["step_count"] + 1}

def should_continue(state: ReActState) -> str:
    if state["final_answer"] or state["step_count"] >= 5:
        return "end"
    return "think"

# Build graph
graph = StateGraph(ReActState)
graph.add_node("think", think_node)
graph.add_node("act", act_node)

graph.set_entry_point("think")
graph.add_edge("think", "act")
graph.add_conditional_edges("act", should_continue, {"think": "think", "end": END})

app = graph.compile()

# Test
result = app.invoke({
    "question": "Who created Python and what framework is built on LangChain for workflows?",
    "thoughts": [], "actions": [], "observations": [],
    "final_answer": "", "step_count": 0
})

print(f"\nFinal answer: {result['final_answer']}")
print(f"Steps taken: {result['step_count']}")

### Demo 4: Implementing Cycles for Iterative Refinement

Cycles allow a graph to loop back and improve its output. This pattern is essential for self-correcting agents that can critique and refine their own work.

In [ ]:
# Demo 4 - Implementing Cycles for Iterative Refinement

class RefinementState(TypedDict):
    topic: str
    draft: str
    feedback: str
    iteration: int
    is_good_enough: bool

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def write_node(state: RefinementState) -> dict:
    """Write or rewrite based on feedback."""
    if state["draft"]:
        prompt = f"Improve this text based on the feedback.\n\nText: {state['draft']}\n\nFeedback: {state['feedback']}\n\nWrite an improved version (2-3 sentences):"
    else:
        prompt = f"Write a 2-3 sentence explanation of: {state['topic']}"
    
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Iteration {state['iteration'] + 1}] Draft: {response.content[:100]}...")
    return {"draft": response.content, "iteration": state["iteration"] + 1}

def critique_node(state: RefinementState) -> dict:
    """Evaluate the draft and provide feedback."""
    response = llm.invoke([
        SystemMessage(content="Rate this text 1-10 for clarity and completeness. If 8+, respond EXACTLY with 'APPROVED'. Otherwise give specific improvement feedback."),
        HumanMessage(content=state["draft"])
    ])
    feedback = response.content
    is_good = "APPROVED" in feedback.upper() or state["iteration"] >= 3
    print(f"  [Critique] {'APPROVED' if is_good else feedback[:80]}...")
    return {"feedback": feedback, "is_good_enough": is_good}

def should_refine(state: RefinementState) -> str:
    if state["is_good_enough"]:
        return "end"
    return "write"

# Build graph with cycle
graph = StateGraph(RefinementState)
graph.add_node("write", write_node)
graph.add_node("critique", critique_node)

graph.set_entry_point("write")
graph.add_edge("write", "critique")
graph.add_conditional_edges("critique", should_refine, {"write": "write", "end": END})

app = graph.compile()

result = app.invoke({
    "topic": "Why LangGraph is useful for building AI agents",
    "draft": "", "feedback": "", "iteration": 0, "is_good_enough": False
})

print(f"\nFinal draft (after {result['iteration']} iterations):")
print(result["draft"])

### Demo 5: Human-in-the-Loop with Checkpointing

In production agents, you often want human oversight before critical actions. LangGraph supports this through interrupt points where the graph pauses and waits for human approval before continuing.

In [ ]:
# Demo 5 - Human-in-the-Loop (Simulated)

# Note: Full checkpointing requires a persistence backend (e.g., SQLite).
# This demo simulates the pattern without external dependencies.

class HumanLoopState(TypedDict):
    task: str
    plan: str
    human_approved: bool
    execution_result: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def plan_node(state: HumanLoopState) -> dict:
    """Generate a plan for the task."""
    response = llm.invoke([
        SystemMessage(content="Create a brief 3-step plan for the task. Be specific."),
        HumanMessage(content=state["task"])
    ])
    print(f"  Plan generated: {response.content[:100]}...")
    return {"plan": response.content}

def human_review_node(state: HumanLoopState) -> dict:
    """Simulate human review (auto-approve for demo)."""
    print(f"  [HUMAN REVIEW] Plan to review:")
    print(f"    {state['plan'][:200]}...")
    # In production, this would pause and wait for human input
    approved = True  # Simulated approval
    print(f"  [HUMAN] {'Approved' if approved else 'Rejected'}")
    return {"human_approved": approved}

def execute_node(state: HumanLoopState) -> dict:
    """Execute the approved plan."""
    if not state["human_approved"]:
        return {"execution_result": "Execution skipped - plan was not approved."}
    
    response = llm.invoke([
        SystemMessage(content="Execute this plan and provide the result."),
        HumanMessage(content=f"Plan: {state['plan']}")
    ])
    return {"execution_result": response.content}

def route_after_review(state: HumanLoopState) -> str:
    return "execute" if state["human_approved"] else "end"

# Build graph
graph = StateGraph(HumanLoopState)
graph.add_node("plan", plan_node)
graph.add_node("human_review", human_review_node)
graph.add_node("execute", execute_node)

graph.set_entry_point("plan")
graph.add_edge("plan", "human_review")
graph.add_conditional_edges("human_review", route_after_review, {
    "execute": "execute", "end": END
})
graph.add_edge("execute", END)

app = graph.compile()

result = app.invoke({
    "task": "Analyze customer feedback data and create a summary report",
    "plan": "", "human_approved": False, "execution_result": ""
})

print(f"\nExecution result: {result['execution_result'][:200]}...")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders. Fill in the code to complete each task.

### Task 1: Build a Simple Linear Workflow with LangGraph

Create a three-node linear graph that takes a raw text input and processes it through: (1) cleaning, (2) summarization, (3) translation.

In [ ]:
# Task 1 - Build a Simple Linear Workflow with LangGraph

# TODO: Create a StateGraph with three nodes:
# 1. clean_node: Remove extra whitespace and normalize the text
# 2. summarize_node: Use the LLM to summarize the cleaned text
# 3. translate_node: Use the LLM to translate the summary to Spanish
#
# Hint: Define a TypedDict state with fields: raw_text, clean_text, summary, translation
# Hint: Use graph.set_entry_point() and graph.add_edge() for linear flow
# Hint: Each node returns a dict with only the fields it updates

# YOUR CODE HERE


# Test your workflow
# result = app.invoke({...})
# print(f"Summary: {result['summary']}")
# print(f"Translation: {result['translation']}")

### Task 2: Create a Conditional Routing Agent

Build a support agent that classifies incoming messages (billing, technical, general) and routes them to specialized handler nodes.

In [ ]:
# Task 2 - Create a Conditional Routing Agent

# TODO: Build a support agent with:
# 1. A classify_node that uses the LLM to categorize the message
# 2. Three handler nodes: billing_node, technical_node, general_node
# 3. Conditional edges that route based on the classification
#
# Hint: Use add_conditional_edges() with a routing function
# Hint: Each handler should use a different system prompt persona
# Hint: All handlers should connect to END

# YOUR CODE HERE


# Test your agent
# result = app.invoke({"message": "I was charged twice for my subscription", ...})
# print(result["response"])

### Task 3: Implement a Self-Correcting Code Generation Workflow

Build a workflow that generates Python code, tests it (simulated), and if it fails, loops back to fix the code — up to 3 attempts.

In [ ]:
# Task 3 - Implement a Self-Correcting Code Generation Workflow

# TODO: Build a cyclic graph with:
# 1. generate_node: LLM generates Python code for the given task
# 2. test_node: Simulated testing (use exec() or simple string checks)
# 3. fix_node: LLM receives the error and fixes the code
# 4. Conditional edge: if tests pass -> END, else -> fix_node -> test_node
# 5. Max 3 iterations to prevent infinite loops
#
# Hint: State should include: task, code, test_result, passed, attempts
# Hint: Use a conditional edge after test_node to decide pass/retry
# Hint: The fix_node should receive the error message as context

# YOUR CODE HERE


# Test your workflow
# result = app.invoke({"task": "Write a function that checks if a number is prime", ...})
# print(f"Code:\n{result['code']}")
# print(f"Passed: {result['passed']} (attempts: {result['attempts']})")

### Task 4: Build a Research Agent with Planning and Execution Nodes

Build an agent that first plans a research strategy, then executes each step, and finally synthesizes the findings into a report.

In [ ]:
# Task 4 - Build a Research Agent with Planning and Execution Nodes

# TODO: Build a multi-stage research agent with:
# 1. plan_node: LLM generates a 3-step research plan (as JSON list)
# 2. execute_node: LLM executes the current step (simulated research)
# 3. check_node: Checks if all steps are complete
# 4. synthesize_node: LLM combines all findings into a final report
#
# Hint: State should include: topic, plan (list), current_step, findings (list), report
# Hint: execute_node processes one step at a time and increments current_step
# Hint: check_node routes back to execute if more steps remain, else to synthesize

# YOUR CODE HERE


# Test your agent
# result = app.invoke({"topic": "The impact of LLMs on software development", ...})
# print(f"Research Report:\n{result['report']}")

---
## Summary

In this session you learned how to orchestrate complex agentic workflows with LangGraph:

1. **StateGraph Basics** -- How to define typed state, add nodes as Python functions, and connect them with edges.
2. **Conditional Routing** -- How to use conditional edges so the graph takes different paths based on LLM decisions.
3. **ReAct Agents** -- How to implement the Reason-Act-Observe loop as a cyclic graph for tool-using agents.
4. **Iterative Refinement** -- How cycles enable self-correcting workflows that improve output quality over multiple passes.
5. **Human-in-the-Loop** -- How to add pause points for human oversight before critical actions.

**Up Next:** In Session 4, we will combine everything into multi-agent architectures — supervisor-worker patterns, agent handoffs, and collaborative problem-solving systems.